# Identifying Suspicious Insurance Claims by State

We want to identify the most suspicious claims in each state. We'll consider the top 5 percentile of claims with the highest fraud scores in each state as potentially fraudulent.

Your output should include the policy number, state, claim cost, and fraud score.

🌀 Trust me, this one will surely challenge you...! You'll learn Mutiple Ctes, Joins. Give it a try and share the output! 👇

In [0]:
%skip
%sql
CREATE TABLE ska_catalog2.bronze.claims (policy_number VARCHAR(50), state VARCHAR(50), claim_cost FLOAT, fraud_score FLOAT);

INSERT INTO ska_catalog2.bronze.claims (policy_number, state, claim_cost, fraud_score) VALUES ('POL123', 'CA', 10000.00, 85.5), ('POL124', 'CA', 5000.00, 70.2), ('POL125', 'CA', 20000.00, 92.8), ('POL126', 'NY', 15000.00, 88.1), ('POL127', 'NY', 8000.00, 65.4), ('POL128', 'NY', 25000.00, 93.7), ('POL129', 'TX', 12000.00, 75.3), ('POL130', 'TX', 18000.00, 95.2), ('POL131', 'TX', 9000.00, 60.0), ('POL132', 'FL', 11000.00, 82.0), ('POL133', 'FL', 14000.00, 87.5), ('POL134', 'FL', 30000.00, 99.0);

In [0]:
%sql
SELECT * FROM ska_catalog2.bronze.claims

In [0]:
%sql
 
 SELECT 
  c.policy_number,
  c.state,
  c.claim_cost,
  c.fraud_score,
  fsp.percentile_95
FROM ska_catalog2.bronze.claims c
JOIN (
  SELECT
   state,
   PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY fraud_score) OVER (PARTITION BY state) AS percentile_95
  FROM ska_catalog2.bronze.claims
  ) AS  fsp
ON c.state = fsp.state
WHERE c.fraud_score >= fsp.percentile_95

In [0]:
%sql
WITH fraudScorePercetile AS (
  SELECT
   state,
   PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY fraud_score) OVER (PARTITION BY state) AS percentile_95
  FROM ska_catalog2.bronze.claims
),
potentialFraudulentClaims As (
  SELECT
    c.policy_number,
    c.state,
    c.claim_cost,
    c.fraud_score
  FROM ska_catalog2.bronze.claims c
  JOIN fraudScorePercetile fsp
    ON c.state = fsp.state
  WHERE c.fraud_score >= fsp.percentile_95
)
SELECT
  policy_number,
  state,
  claim_cost,
  fraud_score
FROM potentialFraudulentClaims
ORDER BY state, fraud_score DESC